In [ ]:
# Import necessary packages for the script
import pandas as pd
import seaborn as sn
import matplotlib.pyplot as plt

In [ ]:
# Load the combined housing + meeting text dataset into a pandas DataFrame (You will need to change the directory)
df = pd.read_csv('/Users/emilymoore/Downloads/DS 4002 Project 1/cleaned_housing_market_data.csv')

In [ ]:
# 1. TIME-SERIES: Uncertainty Score vs. Median Sales Price

# Ensure date is in datetime format for proper plotting
df['date'] = pd.to_datetime(df['date']) # Convert the 'date' column to datetime format to ensure proper time-series plotting
df = df.sort_values('date') # Sort the data chronologically

# Using dual y-axes to compare scale-divergent metrics
plt.figure(figsize=(12, 6)) # Set the frame size of the figure
ax1 = plt.gca() # Get the current axis (primary y-axis)
ax2 = ax1.twinx() # Create a secondary y-axis that shares the same x-axis (uncertainty score and sales price are on different scales)
# Lineplot of the uncertainty score on the y-axis, colored red to differentiate from median sales price
sns.lineplot(data=df, x='date', y='uncertainty_score', ax=ax1, marker='o', color='red', label='Uncertainty Score') 
# Lineplot of the uncertainty score on the y-axis, colored red to differentiate from uncertainty score
sns.lineplot(data=df, x='date', y='median_sales_price', ax=ax2, marker='s', color='blue', label='Median Sales Price')
ax1.set_ylabel('Uncertainty Score', color='red') # Label the uncertainty score line
ax2.set_ylabel('Median Sales Price ($)', color='blue') # Label the median sales price line
plt.title('Uncertainty Score vs. Median Sales Price Over Time') # Give the visualization a title
ax1.legend(loc='upper left') # Add separate legends for each axis to avoid overlap
ax2.legend(loc='upper right') 
plt.savefig('uncertainty_vs_price_timeseries.png') # Save the figure to a PNG file for inclusion in your report

In [ ]:
# 2. CORRELATION HEATMAP

# Identifying which meeting metrics correlate with market metrics
plt.figure(figsize=(9, 7)) # Set the size of the figure
cols = ['uncertainty_score', 'housing_mentions', 'sales', 'new_listings', 
        'median_sales_price', 'price_volatility', 'meeting_duration_min'] # Select the variables to include in the correlation matrix
# Compute the Pearson correlation matrix for the selected columns
sns.heatmap(df[cols].corr(), # .corr() calculates pairwise linear correlations between variables
            annot=True, # Display numerical correlation values inside each cell
            cmap='coolwarm', # Diverging color palette (red = positive, blue = negative)
            fmt=".2f") # Format correlation coefficients to two decimal places
plt.title('Correlation Heatmap: Meeting Metrics vs. Market Metrics') # Add a title to explain the purpose of the visualization
plt.savefig('correlation_heatmap.png') # Save the heatmap as a PNG file for inclusion in your report

In [ ]:
# 3. SCATTER PLOT: Uncertainty vs. Price Volatility
# Does higher uncertainty predict larger price swings?

# Calculate month-over-month percentage change in median sales price
df['price_change'] = df['median_sales_price'].pct_change() # This measures short-run price volatility (Price_t - Price_{t-1}) / Price_{t-1}

# Remove the first observation (it will be NaN due to pct_change)
df_plot = df.iloc[1:].copy() # .copy() prevents SettingWithCopy warnings
df_plot = df_plot[df_plot['uncertainty_score'] > 0] # Keep only months where uncertainty_score > 0

plt.figure(figsize=(8, 6)) # Create a new figure with specific dimensions
sns.regplot(
    data=df_plot, 
    x='uncertainty_score', # Independent variable (policy uncertainty)
    y='price_change', # Dependent variable (monthly price change)
    scatter_kws={'s':100, 'color':'teal'}, # Customize point size and color
    line_kws={'color':'darkorange'} # Customize regression line color
)
plt.axhline(0, color='black', linestyle='--', alpha=0.5) # Add a horizontal reference line at 0, this shows a distinct difference between 
# positive and negative values

plt.title('Uncertainty Score vs. Monthly Price Change') # Add a title
plt.xlabel('Uncertainty Score (from Minutes)') # Add a label on the x-axis
plt.ylabel('Price % Change (Current vs Previous Month)') # Add a label on the y-axis
plt.grid(True, linestyle='--', alpha=0.6) # Add a light grid to improve readability
plt.tight_layout() # # Adjust spacing to prevent label cutoff
plt.savefig('volatility_scatterplot.png')